In [91]:
# This is the same as the main.py
import os
import sys
import math
import h5py

import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy import ndimage
from numpy.typing import NDArray
from typing import List, Tuple


import itk
from src.indi.extensions.image_registration_lowrank import PCA_rank, template_recon, low_rank_denoising

In [92]:
def plot_image_grid(images, b_values, title="Image Grid"):

    images = np.array(images)  # [N, H, W]
    num_images, ny, nx = images.shape
    
    # mask[int((large_dim - short_dim * 1.2) / 2) : int((large_dim + short_dim * 1.2) / 2), :] = 1

    if nx > ny:
        large_dim = nx
        short_dim = ny
        images = images[:, :, int((large_dim - short_dim * 1.2) / 2) : int((large_dim + short_dim * 1.2) / 2)]
    elif ny > nx:
        large_dim = ny
        short_dim = nx
        images = images[:, int((large_dim - short_dim * 1.2) / 2) : int((large_dim + short_dim * 1.2) / 2), :]
    
    vmin = np.percentile(images, 1)
    vmax = np.percentile(images, 99)
    # Determine grid size (rows × cols)
    cols = math.ceil(np.sqrt(num_images))
    rows = math.ceil(num_images / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    axes = axes.flatten()

    for i in range(rows * cols):
        ax = axes[i]
        if i < num_images:

            ax.imshow(images[i], cmap='gray', vmin=vmin, vmax=vmax)            
            ax.axis("off")
            ax.set_title(f"{b_values[i]}", fontsize=10)
        else:
            ax.axis("off")  # hide unused subplot
            

    # fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


def transfer_points(points, def_field):
    # transform the points to a new coordinate system
    '''
    points: [N, 2] array of points
    disp: [H, W, 2] array of displacement field
    '''

    y_coords, x_coords = points[:, 0], points[:, 1]


    dy = ndimage.map_coordinates(
        input = def_field[:, :, 0],
        coordinates = [x_coords, y_coords],
        order = 1,
        mode = 'nearest'
    )
    dx = ndimage.map_coordinates(
        input = def_field[:, :, 1],
        coordinates = [x_coords, y_coords],
        order = 1,
        mode = 'nearest'
    )

    new_points = points - np.stack((dy, dx), axis=-1)
    return new_points


def get_grid_image(img_shape: NDArray, grid_step: int) -> NDArray:
    """

    Get an image with a regular grid. This is going to be deformed by the
    displacement field of the registration

    Parameters
    ----------
    img_shape
    grid_step

    Returns
    -------
    grid image

    """
    grid_img = np.zeros(img_shape)
    for i in range(0, img_shape[0], grid_step):
        grid_img[i, :] = 1
    for j in range(0, img_shape[1], grid_step):
        grid_img[:, j] = 1
    return grid_img

def compose_deformation_fields(def_field1, def_field2):
    # def_field1 is here a deformation field and def_field2 a displacement field
    composed_field = np.zeros((def_field1.shape[0], def_field1.shape[1], 2), dtype=np.float)
    composed_field[:, :, 0] = def_field1[:, :, 0] + warp_image(def_field2[:, :, 0], def_field1) 
    composed_field[:, :, 1] = def_field1[:, :, 1] + warp_image(def_field2[:, :, 1], def_field1)
    return composed_field

def warp_image(floating_image, def_field):
    # Apply the deformation field to the input floating image 
    warped_image = ndimage.map_coordinates(floating_image,
                                            [def_field[:, :, 0], def_field[:, :, 1]], order=1,
                                            cval=0.0,
                                            prefilter=True)
    return warped_image

In [93]:
casenum = 'HV20'
phase = 'systole'
acqui = 'M12'
# load the pedro version then process the data of 
# datapath = os.path.join('/Users/wangfanwen/Projects/cDTIData/FW/', casenum)
pedropath = os.path.join('/Users/wangfanwen/Projects/cDTIData/Pedro', casenum, phase, acqui)
method = ["LowRankH2L", "LowRank2nd"]
FWpath = os.path.join('/Users/wangfanwen/Projects/cDTIData/', method[1])

data_zip_file = os.path.join(pedropath, 'diffusion_images','data.gz')
data = pd.read_pickle(data_zip_file, compression="gzip")

image_path = os.path.join(pedropath, 'diffusion_images', "images.h5")

with h5py.File(image_path, "r") as hf:
    mov_all = np.array(hf["pixel_values"])  # or hf["pixel_values"][:]

# build the new path with the folder and 
session_path = os.path.join(pedropath, 'Python_post_processing', 'session')
reference_path = os.path.join(session_path, 'image_registration_reference_slice_00.npz')
extra_reged_path = os.path.join(session_path, 'image_registration_extras_slice_00.npz')
segpath = os.path.join(session_path, 'manual_lv_segmentation_slice_000.npz')

reged_path = os.path.join(session_path, 'image_registration_data_slice_00.zip')
# 
# reference = np.load(reference_path, allow_pickle=True)['ref_images'].item()
reference = np.load(reference_path, allow_pickle=True)['ref_images'].item()
segment = np.load(segpath, allow_pickle=True)
data_basic = pd.read_pickle(reged_path, compression="zip")
extra_reged = np.load(extra_reged_path, allow_pickle=True)['registration_image_data'].item()
# extra_reged = np.load(extra_reged_path, allow_pickle=True)
mask_3c = segment['mask_3c']
segmentation = segment['segmentation'].item()
epicardium = segmentation['epicardium']
endocardium = segmentation['endocardium']
anterior_ip = segmentation['anterior_ip']
inferior_ip = segmentation['inferior_ip']
slice_idx = 0 

parameter_object = itk.ParameterObject.New()
parameter_object.AddParameterFile(
    os.path.join('/Users/wangfanwen/Projects/INDI_latest/src/indi/', "extensions", "image_registration_recipes", "Elastix_rigid.txt")
)

# create the registration_image_data placeholder
registration_image_data_new = {}
registration_image_data_new["deformation_field"] = {}
registration_image_data_new["deformation_field"]["field"] = np.empty(mov_all.shape + (2,))
registration_image_data_new["deformation_field"]["grid"] = np.empty(mov_all.shape)
registration_image_data_new["img_post_reg"] = np.empty(mov_all.shape)
registration_image_data_new["img_pre_reg"] = np.empty(mov_all.shape)

# define the mask 
image_stack = np.array(mov_all)
# create mask stack of the FOV central region
mask = np.zeros([image_stack.shape[1], image_stack.shape[2]])
n_images = image_stack.shape[0]
if image_stack.shape[1] > image_stack.shape[2]:
    short_dim = image_stack.shape[2]
    large_dim = image_stack.shape[1]
    mask[int((large_dim - short_dim * 1.2) / 2) : int((large_dim + short_dim * 1.2) / 2), :] = 1
else:
    short_dim = image_stack.shape[1]
    large_dim = image_stack.shape[2]
    mask[:, int((large_dim - short_dim * 1.2) / 2) : int((large_dim + short_dim * 1.2) / 2)] = 1
mask_arr = np.asarray(mask, dtype=np.ubyte)
mask_itk = itk.GetImageFromArray(mask_arr)

In [94]:
new_session_path = os.path.join(FWpath, casenum,phase, acqui, 'Python_post_processing', 'session')
if not os.path.exists(new_session_path):
    os.makedirs(new_session_path)

In [99]:
# method 1: groupwise low-rank
# reg takes the reg_object, data_basic, registration_image_data_new, mov_all, mask_itk
def low_rank_reg_all(
    parameter_object: itk.ParameterObject,
    mov_all: np.ndarray,
    mask_itk: itk.Image,
) -> Tuple[np.ndarray, List[np.ndarray], List[np.ndarray], List[np.ndarray]]:
    """
    Register a stack of diffusionweighted images (DWIs) to a
    low-rankdenoised reference using Elastix + Transformix.

    Parameters
    ----------
    parameter_object : itk.ParameterObject
        Elastix parameter object defining the registration strategy.
    mov_all : np.ndarray
        3-D or 4-D array of moving images with shape (N, Z, Y, X).
    mask_itk : itk.Image
        Binary mask (fixed domain) to constrain registration.

    Returns
    -------
    ref : np.ndarray
        The low-rank denoised reference image (fixed image).
    reged_imgs : list[np.ndarray]
        List of registered moving images.
    grid_imgs : list[np.ndarray]
        List of transformed grid images (visual QA).
    disp_fields : list[np.ndarray]
        List of deformation fields (vector fields, float32).
    """
    # --- 1. Low-rank PCA denoising ------------------------------------
    ref, denoised_imgs = low_rank_denoising(mov_all)
    ref_itk = itk.GetImageFromArray(ref.astype(np.float32))

    # --- 2. Prepare outputs ------------------------------------------
    reged_imgs:  list[np.ndarray] = []
    grid_imgs:   list[np.ndarray] = []
    disp_fields: list[np.ndarray] = []

    # --- 3. Register each moving volume ------------------------------
    for i in tqdm(range(mov_all.shape[0]), desc="Registering volumes"):
        mov_itk = itk.GetImageFromArray(denoised_imgs[i].astype(np.float32))

        # 3.1 Rigid / affine / non-linear registration (Elastix)
        _, transform_param = itk.elastix_registration_method(
            ref_itk,
            mov_itk,
            parameter_object=parameter_object,
            fixed_mask=mask_itk,
            log_to_console=False,
        )

        # 3.2 Deformation field
        def_field_itk = itk.transformix_deformation_field(mov_itk, transform_param)
        disp_fields.append(np.asarray(def_field_itk, dtype=np.float32))

        # 3.3 QA grid image
        grid_np = get_grid_image(ref.shape, 10)
        grid_itk = itk.GetImageFromArray(grid_np.astype(np.float32))
        grid_trans_itk = itk.transformix_filter(grid_itk, transform_param)
        grid_trans_np = itk.GetArrayFromImage(grid_trans_itk)
        grid_trans_np = np.where(grid_trans_np >= 0.1, 0.6, 0.0)
        grid_imgs.append(grid_trans_np)

        # 3.4 Apply deformation to original (non-denoised) moving image
        orig_mov_itk = itk.GetImageFromArray(mov_all[i].astype(np.float32))
        reg_itk = itk.transformix_filter(orig_mov_itk, transform_param)
        reged_imgs.append(itk.GetArrayFromImage(reg_itk))

    return ref, reged_imgs, grid_imgs, disp_fields


def low_rank_reg_all_second(parameter_object, registration_image_data, mov_all, mask_itk):
    """
    Register a stack of diffusionweighted images (DWIs) to a
    low-rankdenoised reference using Elastix + Transformix.

    Parameters
    ----------
    parameter_object : itk.ParameterObject
        Elastix parameter object defining the registration strategy.
    mov_all : np.ndarray
        3-D or 4-D array of moving images with shape (N, Z, Y, X).
    mask_itk : itk.Image
        Binary mask (fixed domain) to constrain registration.

    Returns
    -------
    ref : np.ndarray
        The low-rank denoised reference image (fixed image).
    reged_imgs : list[np.ndarray]
        List of registered moving images.
    grid_imgs : list[np.ndarray]
        List of transformed grid images (visual QA).
    disp_fields : list[np.ndarray]
        List of deformation fields (vector fields, float32).
    """
    # --- 1. Low-rank PCA denoising ------------------------------------
    ref, denoised_imgs = low_rank_denoising(mov_all)
    ref_itk = itk.GetImageFromArray(ref.astype(np.float32))

    # --- 2. Prepare outputs ------------------------------------------
    reged_imgs:  list[np.ndarray] = []
    grid_imgs:   list[np.ndarray] = []
    disp_fields: list[np.ndarray] = []

    # --- 3. Register each moving volume ------------------------------
    for i in tqdm(range(mov_all.shape[0]), desc="Registering volumes"):
        mov_itk = itk.GetImageFromArray(denoised_imgs[i].astype(np.float32))

        # 3.1 Rigid / affine / non-linear registration (Elastix)
        _, transform_param = itk.elastix_registration_method(
            ref_itk,
            mov_itk,
            parameter_object=parameter_object,
            fixed_mask=mask_itk,
            log_to_console=False,
        )

        # 3.2 Deformation field
        def_field_itk = itk.transformix_deformation_field(mov_itk, transform_param)
        disp_fields.append(np.asarray(def_field_itk, dtype=np.float32))

        # 3.4 Apply deformation to original (non-denoised) moving image
        orig_mov_itk = itk.GetImageFromArray(mov_all[i].astype(np.float32))
        reg_itk = itk.transformix_filter(orig_mov_itk, transform_param)
        reged_imgs.append(itk.GetArrayFromImage(reg_itk))
    
    ref_new, denoised_imgs_new = low_rank_denoising(np.array(reged_imgs))
    ref_new_itk = itk.GetImageFromArray(ref_new.astype(np.float32))
    reged_imgs_new:  list[np.ndarray] = []
    disp_fields_new: list[np.ndarray] = []
    for i in tqdm(range(len(reged_imgs)), desc="Registering volumes"):
        mov_new_itk = itk.GetImageFromArray(denoised_imgs_new[i].astype(np.float32))

        # 3.1 Rigid / affine / non-linear registration (Elastix)
        _, transform_param = itk.elastix_registration_method(
            ref_new_itk,
            mov_new_itk,
            parameter_object=parameter_object,
            fixed_mask=mask_itk,
            log_to_console=False,
        )

        # 3.2 Deformation field
        def_field_itk = itk.transformix_deformation_field(mov_new_itk, transform_param)
        def_field_new = itk.GetArrayFromImage(def_field_itk)
        # compose the deformation field with the previous one
        # TODO: this is not correct.
        # disp_new = compose_deformation_fields(disp_fields[i].astype(np.float64) , def_field_new.astype(np.float64) )

        disp_fields_new.append(def_field_new )

        # 3.3 QA grid image
        # TODO: this is not correct.
        grid_np = get_grid_image(ref.shape, 10)
        grid_itk = itk.GetImageFromArray(grid_np.astype(np.float32))
        grid_trans_itk = itk.transformix_filter(grid_itk, transform_param)
        grid_trans_np = itk.GetArrayFromImage(grid_trans_itk)
        grid_trans_np = np.where(grid_trans_np >= 0.1, 0.6, 0.0)
        grid_imgs.append(grid_trans_np)

        # 3.4 Apply deformation to original (non-denoised) moving image
        orig_mov_itk = itk.GetImageFromArray(reged_imgs[i].astype(np.float32))
        reg_itk = itk.transformix_filter(orig_mov_itk, transform_param)
        reged_imgs_new.append(itk.GetArrayFromImage(reg_itk))

    return ref_new, reged_imgs_new, grid_imgs, disp_fields_new


# ---------------------------------------------------------------------
def low_rank_high2low_reg_all(
    parameter_object: itk.ParameterObject,
    data: dict,
    mov_all: np.ndarray,
    mask_itk: itk.Image,
) -> Tuple[np.ndarray, List[np.ndarray], List[np.ndarray], List[np.ndarray]]:
    """
    Twostage registration strategy:
    1.  Split volumes into low- and high-b-value groups (b ≤ 50 vs. > 50).
    2.  Register low-b volumes to a low-rank reference (low-b ref).
    3.  Register high-b volumes to the low-b reference **via** an
        intermediate high-b reference aligned to the low-b reference.

    Parameters
    ----------
    parameter_object : itk.ParameterObject
        Elastix registration parameters.
    data : dict
        Must contain key ``'b_value'`` (1-D array of b-values).
    mov_all : np.ndarray
        Image stack with shape (N, Z, Y, X).
    mask_itk : itk.Image
        Mask for registration (applied only to low-b → low-b step).

    Returns
    -------
    ref_low : np.ndarray
        Low-rank denoised reference constructed from low-b volumes.
    reged_all : list[np.ndarray]
        All volumes after registration, in original order.
    grids_all : list[np.ndarray]
        Transformed grid images (visual QA), same ordering.
    disp_all : list[np.ndarray]
        Deformation fields (float32), same ordering.
    """
    # -----------------------------------------------------------------
    # 1. Split indices by b-value threshold
    # -----------------------------------------------------------------
    b_vals = np.asarray(data["b_value"])
    low_idx  = np.where(b_vals <= 50)[0]
    high_idx = np.where(b_vals >  50)[0]

    low_imgs  = mov_all[low_idx]
    high_imgs = mov_all[high_idx]

    # -----------------------------------------------------------------
    # 2. Build low-rank references
    # -----------------------------------------------------------------
    ref_low,  den_low  = low_rank_denoising(low_imgs)
    ref_high, den_high = low_rank_denoising(high_imgs)

    ref_low_itk  = itk.GetImageFromArray(ref_low.astype(np.float32))
    ref_high_itk = itk.GetImageFromArray(ref_high.astype(np.float32))

    # -----------------------------------------------------------------
    # 3. Align high-b reference to low-b reference (one-off)
    # -----------------------------------------------------------------
    img_high2low_itk, trans_high2low = itk.elastix_registration_method(
        ref_low_itk,          # fixed
        ref_high_itk,         # moving
        parameter_object=parameter_object,
        log_to_console=False,
    )

    # The registered high-b reference becomes the fixed image
    fixed_high2low_itk = img_high2low_itk

    # -----------------------------------------------------------------
    # 4. Helper: register a group of images to a given fixed image
    # -----------------------------------------------------------------
    def register_group(
        fixed_itk: itk.Image,
        denoised_group: np.ndarray,
        original_group: np.ndarray,
        use_mask: bool = False,
    ):
        """Return registered imgs, grids, and deformation fields."""
        reg_imgs, grid_imgs, disp_fields = [], [], []
        for vol in denoised_group:
            mov_itk = itk.GetImageFromArray(vol.astype(np.float32))
            _, trans_param = itk.elastix_registration_method(
                fixed_itk,
                mov_itk,
                parameter_object=parameter_object,
                fixed_mask=mask_itk if use_mask else None,
                log_to_console=False,
            )

            # deformation field
            def_itk = itk.transformix_deformation_field(mov_itk, trans_param)
            disp_fields.append(np.asarray(def_itk, dtype=np.float32))

            # grid image for QA
            grid_np  = get_grid_image(vol.shape, 10)
            grid_itk = itk.GetImageFromArray(grid_np.astype(np.float32))
            grid_tr  = itk.transformix_filter(grid_itk, trans_param)
            grid_np  = np.where(itk.GetArrayFromImage(grid_tr) >= 0.1, 0.6, 0.0)
            grid_imgs.append(grid_np)

            # apply to original (non-denoised) volume
            orig_itk = itk.GetImageFromArray(original_group[len(reg_imgs)].astype(np.float32))
            reg_itk  = itk.transformix_filter(orig_itk, trans_param)
            reg_imgs.append(itk.GetArrayFromImage(reg_itk))

        return reg_imgs, grid_imgs, disp_fields

    # -----------------------------------------------------------------
    # 5. Register low-b group to low-b reference (mask applied)
    # -----------------------------------------------------------------
    reg_low, grid_low, disp_low = register_group(
        ref_low_itk, den_low, low_imgs, use_mask=True
    )

    # -----------------------------------------------------------------
    # 6. Register high-b group to *aligned* high-b reference (no mask)
    # -----------------------------------------------------------------
    reg_high, grid_high, disp_high = register_group(
        fixed_high2low_itk, den_high, high_imgs, use_mask=False
    )

    # -----------------------------------------------------------------
    # 7. Re-assemble results in original order
    # -----------------------------------------------------------------
    reg_concat  = reg_low  + reg_high
    grid_concat = grid_low + grid_high
    disp_concat = disp_low + disp_high

    order = np.concatenate([low_idx, high_idx])
    sort_idx = np.argsort(order)

    reged_all  = [reg_concat[i]  for i in sort_idx]
    grids_all  = [grid_concat[i] for i in sort_idx]
    disp_all   = [disp_concat[i] for i in sort_idx]

    return ref_low, reged_all, grids_all, disp_all


# ref, reged_imgs_group, grids, disp = low_rank_reg_all(
#     parameter_object, data_basic, registration_image_data_new, mov_all, mask_itk
# )
# ref, reged_imgs_group, grids, disp = low_rank_high2low_reg_all(
#     parameter_object, data, registration_image_data_new, mov_all, mask_itk
# )

ref, reged_imgs_group, grids, disp = low_rank_reg_all_second(
    parameter_object, data, mov_all, mask_itk
)

# save the registration_image_data_new
registration_image_data_new["deformation_field"]["field"] = disp
registration_image_data_new["deformation_field"]["grid"] = grids
registration_image_data_new["img_post_reg"] = np.array(reged_imgs_group)
registration_image_data_new["img_pre_reg"] = mov_all

Registering volumes: 100%|██████████| 84/84 [00:24<00:00,  3.46it/s]


In [100]:

# This saves the reg
data["image"] = reged_imgs_group
reg_file = os.path.join(
    new_session_path, "image_registration_data_slice_" + str(slice_idx).zfill(2) + ".zip"
)
data_basic.to_pickle(reg_file, compression={"method": "zip", "compresslevel": 9})

# This saves the extra reg
data_basic["image"] = None
data_basic["image"] = reged_imgs_group
reg_file_extras = os.path.join(
    new_session_path, "image_registration_extras_slice_" + str(slice_idx).zfill(2) + ".npz"
)
np.savez_compressed(
    reg_file_extras,
    registration_image_data=registration_image_data,
)

# This saves reference
ref_image_new = {}
ref_image_new["image"] = ref
ref_image_new["index"] = 0
ref_image_new["n_images"] = mov_all.shape[0]
ref_image_new["groupwise_reg_info"] = {}
new_refer_file = os.path.join(new_session_path, "image_registration_reference_slice_" + str(slice_idx).zfill(2) + ".npz")
np.savez_compressed(new_refer_file, ref_images=ref_image_new)

In [106]:
# take the average of the old and new one.
old_ref = np.mean(extra_reged["img_post_reg"], axis=0)
new_ref = np.mean(np.array(reged_imgs_group), axis=0)
# shift the old_ref along axis 0
# new_ref = np.roll(old_ref, shift=20, axis=0)

old_ref_itk = itk.GetImageFromArray(old_ref.astype(np.float32, copy=False))
new_ref_itk = itk.GetImageFromArray(new_ref.astype(np.float32, copy=False))

img_reg, result_transform_parameters = itk.elastix_registration_method(
    new_ref_itk,# fixed
    old_ref_itk, 
    parameter_object=parameter_object,
    fixed_mask=mask_itk,
    log_to_console=True,
)

binary_parameters = result_transform_parameters
binary_parameters.SetParameter(0, "ResampleInterpolator", ["FinalNearestNeighborInterpolator"])
binary_parameters.SetParameter(0, "FinalBSplineInterpolationOrder", ["0"])

mask_3c_itk = itk.GetImageFromArray(mask_3c)
mask_itk_new = itk.transformix_filter(mask_3c_itk, binary_parameters)

def_field = itk.transformix_deformation_field(new_ref_itk, result_transform_parameters)
def_field = np.asarray(def_field).astype(np.float32)
# get the disp from the deformation field

# transfer the points
epicardium_new = transfer_points(epicardium, def_field)
endocardium_new = transfer_points(endocardium, def_field)
anterior_ip_old = anterior_ip[np.newaxis, :]
anterior_ip_new = transfer_points(anterior_ip_old, def_field)
inferior_ip_old = inferior_ip[np.newaxis, :]
inferior_ip_new = transfer_points(inferior_ip_old, def_field)
# get the disp from the 
inferior_ip_new = inferior_ip_new[0]
anterior_ip_new = anterior_ip_new[0]

# save them to a new segmentation file
new_segpath = os.path.join(new_session_path, 'manual_lv_segmentation_slice_' + str(slice_idx).zfill(3) + '.npz')
mask_3c_new = itk.GetArrayFromImage(mask_itk_new)

segmentation_new = {}
segmentation_new['epicardium'] = epicardium_new
segmentation_new['endocardium'] = endocardium_new
segmentation_new['anterior_ip'] = anterior_ip_new
segmentation_new['inferior_ip'] = inferior_ip_new

np.savez_compressed(
    os.path.join(new_session_path, "manual_lv_segmentation_slice_" + str(slice_idx).zfill(3) + ".npz"),
    mask_3c=mask_3c_new,
    segmentation=segmentation_new,
)

ELASTIX version: 5.2.0
Command line options from ElastixBase:
-threads  unspecified, so all available threads are used
Command line options from TransformBase:
-t0       unspecified, so no initial transform used

Reading images...
Reading images took 0 ms.

  A default pyramid schedule is used.
  A default pyramid schedule is used.
  The default value "GeometricalCenter" is used instead.
Transform parameters are initialized as: [0, 0, 0]
Scales are estimated automatically.
Scales for transform parameters are: [6229.166666666667, 1, 1]
Initialization of all components (before registration) took: 3 ms.
Preparation of the image pyramids took: 3 ms.

Resolution: 0
but the selected ImageSampler is not suited for that.
  The default value "false" is used instead.
  The default value "true" is used instead.
  The default value "true" is used instead.
  The default value "false" is used instead.
Setting the fixed masks took: 0 ms.
Setting the moving masks took: 0 ms.
  The default value "20" i

In [107]:

import shutil
# copy the image_manual_removal_post.zip, image_manual_removal_pre.zip to the new path
shutil.copy(os.path.join(session_path, 'image_manual_removal_post.zip'), new_session_path)
shutil.copy(os.path.join(session_path, 'image_manual_removal_pre.zip'), new_session_path)
src_dir = os.path.join(pedropath, 'diffusion_images')
dst_dir = os.path.join(FWpath, casenum, phase, acqui, 'diffusion_images')

if os.path.exists(dst_dir):
    shutil.rmtree(dst_dir)

shutil.copytree(src_dir, dst_dir)

'/Users/wangfanwen/Projects/cDTIData/LowRank2nd/HV20/systole/M12/diffusion_images'

In [ ]:
print(extra_reged.keys())

In [ ]:
# plot the registered mean image\
ms = 2          # marker size in points – shrink or grow as you like

plt.subplot(1, 3, 1)
plt.imshow(mask_3c, cmap='gray')
plt.plot(epicardium[:, 0], epicardium[:, 1], color='r', linestyle='-', marker='.', markersize=ms)
plt.plot(endocardium[:, 0], endocardium[:, 1], color='g', linestyle='-', marker='.', markersize=ms)
plt.plot(anterior_ip[0],  anterior_ip[1],  color='b', linestyle='-', marker='.', markersize=ms)
plt.plot(inferior_ip[0],  inferior_ip[1],  color='y', linestyle='-', marker='.', markersize=ms)
plt.title('Source')

plt.subplot(1, 3, 2)
plt.imshow(mask_3c_new, cmap='gray')
plt.plot(epicardium_new[:, 0], epicardium_new[:, 1], color='r', linestyle='-', marker='.', markersize=ms)
plt.plot(endocardium_new[:, 0], endocardium_new[:, 1], color='g', linestyle='-', marker='.', markersize=ms)
plt.plot(anterior_ip_new[0],  anterior_ip_new[1],  color='b', linestyle='-', marker='.', markersize=ms)
plt.plot(inferior_ip_new[0],  inferior_ip_new[1],  color='y', linestyle='-', marker='.', markersize=ms)
plt.title("Moved")



In [ ]:
ref_itk = itk.GetImageFromArray(ref)
mask = itk.GetImageFromArray(mask_arr)

reged_imgs_group = []
for i in tqdm(range(ref_image_new["n_images"])):
    denoised_mov = itk.GetImageFromArray(denoised_img[i])

    img_reg, result_transform_parameters = itk.elastix_registration_method(
        ref_itk,
        denoised_mov,
        parameter_object=parameter_object,
        fixed_mask=mask,
        log_to_console=False,
    )
    def_field = itk.transformix_deformation_field(denoised_mov, result_transform_parameters)
    def_field_np = np.asarray(def_field).astype(np.float32)
    registration_image_data["deformation_field"]["field"][i] = def_field_np

    grid_img = get_grid_image(ref.shape, 10)
    grid_img_itk = itk.GetImageFromArray(grid_img)
    grid_img_transformed = itk.transformix_filter(grid_img_itk, result_transform_parameters)
    grid_img_transformed_np = itk.GetArrayFromImage(grid_img_transformed)
    grid_img_transformed_np[grid_img_transformed_np < 0.1] = 0
    grid_img_transformed_np[grid_img_transformed_np >= 0.1] = 0.6
    registration_image_data["deformation_field"]["grid"][i] = grid_img_transformed_np
    registration_image_data["img_pre_reg"][i] = mov_all[i]
    
    mov_itk = itk.GetImageFromArray(mov_all[i])
    
    img_reg_itk = itk.transformix_filter(mov_itk, result_transform_parameters)
    img_reg = itk.GetArrayFromImage(img_reg_itk)
    registration_image_data["img_post_reg"][i] = img_reg
    
    reged_imgs_group.append(img_reg)

In [ ]:
# This saves the registered images
data["image"] = reged_imgs_group
reg_file = os.path.join(
    new_session_path, "image_registration_data_slice_" + str(slice_idx).zfill(2) + ".zip"
)

# data_basic = data[["file_name", "image", "acquisition_date"]]
data_basic["image"] = None
data_basic["image"] = reged_imgs_group

data_basic.to_pickle(reg_file, compression={"method": "zip", "compresslevel": 9})


# save the registration extras
reg_file_extras = os.path.join(
    new_session_path, "image_registration_extras_slice_" + str(slice_idx).zfill(2) + ".npz"
)

np.savez_compressed(
    reg_file_extras,
    registration_image_data=registration_image_data,
)

In [ ]:
# regenerate the seges
segmentation.keys()

# transfer the 
print(epicardium.shape, endocardium.shape, anterior_ip.shape, inferior_ip.shape)

In [ ]:
# save them ba
plt.subplot(1, 4, 1)
plt.imshow(mask_3c, cmap='gray')
plt.title("Original Mask")
plt.plot(epicardium[:,0], epicardium[:,1], 'r.-')  # [x, y]
plt.plot(endocardium[:,0], endocardium[:,1], 'g.-')  # [x, y]
plt.plot(anterior_ip[0], anterior_ip[1], 'b.-')  # [x, y]
plt.plot(inferior_ip[0], inferior_ip[1], 'y.-')  # [x, y]

plt.subplot(1, 4, 2)
plt.imshow(itk.GetArrayFromImage(mask_itk_new), cmap='gray')
plt.plot(epicardium_new[:,0], epicardium_new[:,1], 'r.-')  # [x, y]
plt.plot(endocardium_new[:,0], endocardium_new[:,1], 'g.-')  # [x, y]
plt.plot(anterior_ip_new[0], anterior_ip_new[1], 'b.-')  # [x, y]
plt.plot(inferior_ip_new[0], inferior_ip_new[1], 'y.-')  # [x, y]
plt.title("Transformed Mask")

# plt.plot(epicardium[:,0], epicardium[:,1], 'g.-')  # [x, y]
plt.subplot(1, 4, 3)
plt.imshow(new_ref, cmap='gray')
plt.title("new ref")

plt.subplot(1, 4, 4)
plt.imshow(old_ref, cmap='gray')
plt.title("old ref")

In [ ]:
print(extra_reged['img_post_reg'].shape)

In [ ]:
extra_reged['img_post_reg'].shape

In [ ]:
ref, denoised_img = low_rank_denoising(mov_all)
b_values = data['b_value']

In [ ]:
# if the bvalue is smaller than 100, put them in one group, otherwise in another
# get the index
low_b_value_indices = np.where(b_values <= 50)[0]
high_b_value_indices = np.where(b_values > 50)[0]

print(low_b_value_indices, high_b_value_indices)

In [ ]:
low_b_value_images = mov_all[low_b_value_indices]
low_b_values = b_values[low_b_value_indices]
high_b_value_images = mov_all[high_b_value_indices]
high_b_values = b_values[high_b_value_indices]
# b_values
ref_high, denoised_high = low_rank_denoising(high_b_value_images)
ref_low, denoised_low = low_rank_denoising(low_b_value_images)

In [ ]:
ref_old = reference['image']
# shift the reference['image']

ref_shifted = np.roll(ref_old, shift=10, axis=0)  # Shift the image by 10 pixels along the x-axis
# 
plt.subplot(1, 2, 1)
plt.imshow(ref_old, cmap='gray')
plt.subplot(1, 2, 2)
plt.imshow(ref_shifted, cmap='gray')

In [ ]:
# reg the low b-value images to the reference image
ref_low_itk = itk.GetImageFromArray(ref_low)
# TODO: revise
old_ref_itk = itk.GetImageFromArray(ref_shifted)
mask_itk = itk.GetImageFromArray(mask_3c)

img_reg, result_transform_parameters = itk.elastix_registration_method(
    old_ref_itk,
    ref_low_itk,
    parameter_object=parameter_object,
    fixed_mask= itk.GetImageFromArray(mask_arr),
    log_to_console=False,
)

In [ ]:
# This is a ParameterObject
binary_parameters = result_transform_parameters

# Set interpolation to NearestNeighbor
# binary_parameters.SetParameter("TransformParameters.0", "FinalBSplineInterpolationOrder", ["0"])
# binary_parameters.SetParameter("TransformParameters.0", "ResampleInterpolator", ["FinalNearestNeighborInterpolator"])

binary_parameters.SetParameter(0, "ResampleInterpolator", ["FinalNearestNeighborInterpolator"])
binary_parameters.SetParameter(0, "FinalBSplineInterpolationOrder", ["0"])

mask_itk_new = itk.transformix_filter(mask_itk, binary_parameters)

# apply this to the mask 

In [ ]:
def_field = itk.transformix_deformation_field(ref_low_itk, result_transform_parameters)
def_field = np.asarray(def_field).astype(np.float32)

In [ ]:
plt.subplot(1, 3, 1)
plt.imshow(ref_low, cmap='gray', vmin = np.percentile(ref_low, 1), vmax = np.percentile(ref_low, 99))
plt.title("Low B-value Reference Image")
plt.subplot(1, 3, 2)
plt.imshow(ref_high, cmap='gray', vmin = np.percentile(ref_high, 1), vmax = np.percentile(ref_high, 99))
plt.title("High B-value Reference Image")
plt.subplot(1, 3, 3)
plt.imshow(ref_shifted, cmap='gray', vmin = np.percentile(reference['image'], 1), vmax = np.percentile(reference['image'], 99))
plt.title("Reference Image")

In [ ]:
plt.subplot(1, 3, 1)
plt.imshow(mask_3c, cmap='gray')
plt.title("Original Mask")
plt.plot(epicardium[:,0], epicardium[:,1], 'r.-')  # [x, y]
plt.plot(endocardium[:,0], endocardium[:,1], 'g.-')  # [x, y]
plt.plot(anterior_ip[0], anterior_ip[1], 'b.-')  # [x, y]
plt.plot(inferior_ip[0], inferior_ip[1], 'y.-')  # [x, y]

plt.subplot(1, 3, 2)
plt.imshow(itk.GetArrayFromImage(mask_itk_new), cmap='gray')
plt.plot(new_points[:,0], new_points[:,1], 'r.-')  # [x, y]
# plt.plot(epicardium[:,0], epicardium[:,1], 'g.-')  # [x, y]
plt.title("Transformed Mask")
plt.subplot(1, 3, 3)
plt.imshow(np.abs(itk.GetArrayFromImage(mask_itk)-mask_itk_new), cmap='gray')
plt.title("Original Mask ITK")


In [ ]:
# old method 
reged_imgs_old = []

for i in range(denoised_img.shape[0]):
    
    mov = np.asarray(mov_all[i], dtype=np.float32)
    denoised_mov = itk.GetImageFromArray(denoised_img[i])

    img_reg, result_transform_parameters = itk.elastix_registration_method(
        ref,
        denoised_mov,
        parameter_object=parameter_object,
        fixed_mask= itk.GetImageFromArray(mask_arr),
        log_to_console=False,
    )
    # def_field = itk.transformix_deformation_field(denoised_mov, result_transform_parameters)
    # def_field = np.asarray(def_field).astype(np.float32)
    mov = itk.GetImageFromArray(mov)
    img_reg = itk.transformix_filter(mov, result_transform_parameters)
    img_reg = itk.GetArrayFromImage(img_reg)
    reged_imgs_old.append(img_reg)

In [ ]:
plot_image_grid(reged_imgs_old, b_values.to_list(), title="B-value Images")

In [ ]:
# plot the axial and 
reged_imgs_arr = np.array(reged_imgs_old)
ns, nx, ny = reged_imgs_arr.shape
plt.subplot(1, 2, 1)
plt.imshow(reged_imgs_arr[:, int(nx//2)-10, :], cmap='gray')
plt.title("First/2")
plt.subplot(1, 2, 2)
plt.imshow(reged_imgs_arr[:, :, int(ny//2)+10], cmap='gray')
plt.title("Second/2")

In [ ]:
# new method
# generate among the low_b, high_b_value, then align one to another
reged_imgs_low, reged_imgs_high = [],[]
# align the high to low b_value

# reg the ref_low and ref_high images
ref_low = itk.GetImageFromArray(ref_low)
ref_high = itk.GetImageFromArray(ref_high)

img_reg_high2low, result_transform_parameters_high2low = itk.elastix_registration_method(
    ref_low,
    ref_high,
    parameter_object=parameter_object,
    # fixed_mask=
    log_to_console=False,
)

# reg the low to ref_low
for i in range(denoised_low.shape[0]):
    denoised_mov = itk.GetImageFromArray(denoised_low[i])

    img_reg, result_transform_parameters = itk.elastix_registration_method(
        ref_low,
        denoised_mov,
        parameter_object=parameter_object,
        log_to_console=False,
    )

    mov = itk.GetImageFromArray(low_b_value_images[i])
    img_reg = itk.transformix_filter(mov, result_transform_parameters)
    img_reg = itk.GetArrayFromImage(img_reg)
    reged_imgs_low.append(img_reg)

for i in range(denoised_high.shape[0]):
    denoised_mov = itk.GetImageFromArray(denoised_high[i])
    
    img_reg, result_transform_parameters = itk.elastix_registration_method(
        img_reg_high2low,
        denoised_mov,
        parameter_object=parameter_object,
        log_to_console=False,
    )

    mov = itk.GetImageFromArray(high_b_value_images[i])
    img_reg = itk.transformix_filter(mov, result_transform_parameters)
    img_reg = itk.GetArrayFromImage(img_reg)
    reged_imgs_high.append(img_reg)

In [ ]:
reged_imgs_new = reged_imgs_low + reged_imgs_high

In [ ]:
# sort the reged_imgs using index
index = np.concatenate((low_b_value_indices, high_b_value_indices))
reged_imgs_new_back = [reged_imgs_new[i] for i in np.argsort(index)]
plot_image_grid(reged_imgs_new_back, b_values.to_list(), title="B-value Images")

In [ ]:
# plot the axial and 
reged_imgs_arr = np.array(reged_imgs_new_back)
ns, nx, ny = reged_imgs_arr.shape
plt.subplot(1, 2, 1)
plt.imshow(reged_imgs_arr[:, int(nx//2)-10, :], cmap='gray')
plt.title("First/2")
plt.subplot(1, 2, 2)
plt.imshow(reged_imgs_arr[:, :, int(ny//2)+10], cmap='gray')
plt.title("Second/2")

In [ ]:
# old method 
reged_imgs_old_2 = []

reged_imgs_old = np.array(reged_imgs_old)

ref2, denoised_img = low_rank_denoising(reged_imgs_old)

ref2 = itk.GetImageFromArray(ref2)

for i in range(denoised_img.shape[0]):
    
    mov = np.asarray(mov_all[i], dtype=np.float32)
    denoised_mov = itk.GetImageFromArray(denoised_img[i])

    img_reg, result_transform_parameters = itk.elastix_registration_method(
        ref2,
        denoised_mov,
        parameter_object=parameter_object,
        # fixed_mask=
        log_to_console=False,
    )
    # def_field = itk.transformix_deformation_field(denoised_mov, result_transform_parameters)
    # def_field = np.asarray(def_field).astype(np.float32)
    mov = itk.GetImageFromArray(mov)
    img_reg = itk.transformix_filter(mov, result_transform_parameters)
    img_reg = itk.GetArrayFromImage(img_reg)
    reged_imgs_old_2.append(img_reg)

In [ ]:
# plot the axial and 
reged_imgs_arr = np.array(reged_imgs_old_2)
ns, nx, ny = reged_imgs_arr.shape
plt.subplot(1, 2, 1)
plt.imshow(reged_imgs_arr[:, int(nx//2)-10, :], cmap='gray')
plt.title("First/2")
plt.subplot(1, 2, 2)
plt.imshow(reged_imgs_arr[:, :, int(ny//2)+10], cmap='gray')
plt.title("Second/2")

In [ ]:
plot_image_grid(reged_imgs_old_2, b_values.to_list(), title="B-value Images")